# Source galaxy counts: Buzzard vs DES Y3

The source clustering bias $b_{g,s}$ is fixed by matching the **pixel histogram of the metacal source counts** to a
reference observation, which absorbs the cosmology dependence of the one-point source distribution into the bias
(section 3 of [Thomsen et al. 2025](https://arxiv.org/abs/2511.04681), fit in `notebooks/sc_bias_fit_count.ipynb`).
That reference changed between the two papers:

| | reference observation | bias table |
|---|---|---|
| paper 1 | Buzzard member 0 | `data/deprecated/metacal_biases_buzzard_0.h5` |
| paper 2 | DES Y3 | `data/desy3_metacal_bias.h5`, group `contam` |

This notebook puts the two references side by side on the one statistic the fit actually sees. The CosmoGrid mocks
follow the DES data by construction, since that is what their bias was fit to, so they enter as the third curve that
shows how well the match is made. `count_maps_desy3.ipynb` covers the DES / CosmoGrid pair in the map domain; what is
added here is Buzzard, read from the 2026-09-26 flock that `notebooks/build_buzzard_flock.ipynb` consumes.

## Buzzard needs two transformations before it can be compared

**Contaminate it.** The DES Y3 source counts are inhomogeneous over the footprint, and the imaging systematics
(depth, seeing, stellar density, ...) widen the observed pixel distribution well beyond what clustering does, most in
the highest tomographic bin. CosmoGrid is contaminated with the `lss_sys` map on the model side before the fit,
precisely so that the bias does not absorb them. Buzzard is a simulation too and just as clean, so comparing it to
DES without the same imprint compares two different things.

**Renormalize it.** The flock carries ~8.4% more sources per bin than `survey.metacal.n_gal`, since its source
selection targets absolute per-bin totals fixed before the v16 recomputation of `n_gal`. A count histogram is not
invariant under that, so the counts are binomially thinned to the DES mean first. Thinning a Poisson field gives
exactly a Poisson field of the reduced mean, so it moves $\bar{n}$ without touching the shot noise budget, which
multiplying by 0.92 would not. `scb.build_target` also asserts $\bar{n}$ against the config to within 5%, so the raw
flock cannot be used as a fit target at all.

Run this on a compute node with the `torch_env` kernel. The 15 flock members are ~50 MB of count maps each and the
four bias fits at the end read the fiducial CosmoGrid maps.

### Config and inputs

In [ ]:
%load_ext autoreload
%autoreload 2

import os

import numpy as np
import h5py
import matplotlib.pyplot as plt
from scipy.stats import wasserstein_distance

from msfm.utils import files, catalog, clustering, buzzard, imports, source_clustering_bias as scb

hp = imports.import_healpy()

In [ ]:
repo_dir = "/users/athomsen/dlss/repos/multiprobe-simulation-forward-model"
conf = files.load_config(os.path.join(repo_dir, "configs/v18/default.yaml"))

# the flock of notebooks/build_buzzard_flock.ipynb, one file per member carrying both probes
FLOCK_DIR = "/capstor/store/cscs/swissai/a0158/athomsen/deep_lss/data/v18/buzzard_flock"
MEMBERS = [0, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
LENS_J = [0, 3, 3, 4, 4, 5, 5, 6, 6, 7, 7, 8, 8, 11, 11]
LENS_K = ["a"] + 7 * ["a", "b"]
flock_files = {
    member: f"{FLOCK_DIR}/DESY3_mock_observation_Buzzard_{j}_Y3{k}_v18.h5"
    for member, j, k in zip(MEMBERS, LENS_J, LENS_K)
}

# the CosmoGrid tree's own fiducial/cosmo_fiducial is empty on scratch, this is the restored copy
fiducial_dir = "/capstor/store/cscs/swissai/a0158/athomsen/deep_lss/data/fiducial/cosmo_fiducial"
bias_file = os.path.join(repo_dir, conf["files"]["metacal_bias"])
bias_file_paper_1 = os.path.join(repo_dir, "data/deprecated/metacal_biases_buzzard_0.h5")

z_bins = conf["survey"]["metacal"]["z_bins"]
n_z = len(z_bins)
n_side = conf["analysis"]["n_side"]
n_pix = hp.nside2npix(n_side)

centres = scb._bin_centres_1pt()
colors = {
    "DES Y3": "black",
    "CosmoGrid": "C2",
    "Buzzard raw": "0.6",
    "Buzzard thinned": "C0",
    "Buzzard contaminated": "C1",
}

### The DES Y3 reference and the contamination map

`scb.build_target` restricts the cached DES Y3 count map to the footprint and histograms each tomographic bin into
200 bins covering 1 to 200 galaxies per pixel, so empty pixels are excluded. `files.get_metacal_systematics` returns
the `contamination` dataset $\langle 1/W \rangle$, which is what a *clean model* is multiplied by, rotated into the
footprint frame and renormalized to unit mean over the base patch. Its pixel order is the base patch, which is
`np.arange(n_pix)[mask]` by construction, and the assertion below is what keeps Buzzard, DES and the map on the same
pixels.

In [ ]:
_, des_count_map = catalog.build_metacal_map_from_cat(conf, debug=False)
target, n_bar, mask = scb.build_target(conf, count_map=des_count_map)
des_counts = des_count_map[mask]

contamination = files.get_metacal_systematics(conf)
_, patches_pix_dict, _, _ = files.load_pixel_file(conf)
assert np.array_equal(patches_pix_dict["metacal"][0][0], np.arange(n_pix)[mask]), "base patch is not the mask"

print(f"footprint = {mask.sum()} pixels")
print("DES Y3 n_bar   =", np.round(n_bar, 3))
print("DES Y3 std     =", np.round(des_counts.std(axis=0), 3))
print("contamination: " + "  ".join(f"{z} std {contamination[:, i].std():.4f}" for i, z in enumerate(z_bins)))

### Buzzard: thin to the DES mean, then contaminate

`thin_to_des` draws `Binomial(N, n_bar_DES / n_bar_Buzzard)` per pixel. `contaminate` routes the thinned counts
through `clustering.galaxy_density_to_count` with $b = 1$ and $\delta = N / \bar{n} - 1$, which reduces to
$N \times \langle 1/W \rangle$ followed by the pipeline's own clip and renormalization. Writing it that way rather
than as a bare multiplication is deliberate, so that the diagnostic follows the forward model if that convention ever
changes. Nothing clips here (counts are non-negative), and the mean drifts by less than 0.03%, since
$\langle 1/W \rangle$ has unit mean over pixels but galaxies are not distributed uniformly over them.

**The contaminated counts are not Poisson resampled.** Buzzard's counts are counts of discrete ADDGALS galaxies, so
they already carry one layer of shot noise, exactly as the DES counts do. Resampling would add a second layer and
inflate every variance by $\bar{n} \approx 72$, which the table further down quotes as the `+ Poisson` row to bound
how much that choice is worth.

In [ ]:
def thin_to_des(counts, seed):
    """Binomially thin footprint counts to the DES Y3 mean, exact for a Poisson field."""
    keep = n_bar / counts.mean(axis=0)
    return np.random.default_rng(seed).binomial(counts.astype(np.int64), keep).astype(np.float32), keep


def contaminate(counts):
    """Impose the DES imaging systematics on clean simulated counts, as the forward model does."""
    return np.stack(
        [
            clustering.galaxy_density_to_count(
                n_bar[i], counts[:, i] / n_bar[i] - 1, 1.0, contamination_map=contamination[:, i]
            )
            for i in range(n_z)
        ],
        axis=-1,
    )


def one_point(counts):
    """(n_z, N_BINS_1PT) pixel count histograms and the Wasserstein distance of each to the DES Y3 target."""
    stats = np.stack([scb._count_statistic(counts[:, i], "1pt", i) for i in range(n_z)])
    distances = np.array(
        [wasserstein_distance(centres, centres, u_weights=stats[i], v_weights=target[i]) for i in range(n_z)]
    )
    return stats, distances


buzzard_arms = {arm: {"stat": [], "mean": [], "std": [], "wd": []} for arm in ("raw", "thinned", "contaminated")}
buzzard_pois = {"mean": [], "std": [], "wd": []}

for member, flock_file in flock_files.items():
    raw = buzzard.get_metacal_counts(flock_file)[mask]
    thinned, keep = thin_to_des(raw, seed=member)
    contaminated = contaminate(thinned)
    # the double counted variant, quoted only to bound what leaving the extra Poisson layer out is worth
    resampled = np.random.default_rng(member).poisson(contaminated).astype(np.float32)

    for arm, counts in (("raw", raw), ("thinned", thinned), ("contaminated", contaminated)):
        stats, distances = one_point(counts)
        buzzard_arms[arm]["stat"].append(stats)
        buzzard_arms[arm]["mean"].append(counts.mean(axis=0))
        buzzard_arms[arm]["std"].append(counts.std(axis=0))
        buzzard_arms[arm]["wd"].append(distances)
    buzzard_pois["mean"].append(resampled.mean(axis=0))
    buzzard_pois["std"].append(resampled.std(axis=0))
    buzzard_pois["wd"].append(one_point(resampled)[1])

    print(f"member {member:2d}  n_bar {np.array2string(raw.mean(axis=0), precision=2)}"
          f"  thinned by {np.array2string(keep, precision=4)}")

for arm in buzzard_arms:
    buzzard_arms[arm] = {key: np.array(value) for key, value in buzzard_arms[arm].items()}
buzzard_pois = {key: np.array(value) for key, value in buzzard_pois.items()}

### CosmoGrid at the fiducial cosmology

Both arms of `sc_bias_fit_count.ipynb` are regenerated from their stored fiducial bias, so the `contam` curve is
literally what the v18 forward model draws its shape noise from. The `clean` one is kept as the reminder of what the
fit does when the systematics have nowhere to go but into $b$.

In [ ]:
delta = scb.read_density_contrast(conf, fiducial_dir)

with h5py.File(bias_file, "r") as f:
    bias_fiducial = {group: f[group]["fiducial"][:] for group in ("clean", "contam")}
    labels = {group: str(f[group].attrs["systematics_label"]) for group in bias_fiducial}
with h5py.File(bias_file_paper_1, "r") as f:
    bias_paper_1 = f["fiducial"][:]

cosmogrid = {}
for group, bias in bias_fiducial.items():
    contam = contamination if group == "contam" else None
    counts = np.stack(
        [
            scb.counts_from_bias(
                n_bar[i], delta[mask, i], bias[i], contamination=None if contam is None else contam[:, i]
            )
            for i in range(n_z)
        ],
        axis=-1,
    )
    stats, distances = one_point(counts)
    cosmogrid[group] = {"mean": counts.mean(axis=0), "std": counts.std(axis=0), "stat": stats, "wd": distances}
    print(f"{group:7s} (systematics {labels[group]!r}): b = {np.round(bias, 4)}")
print(f"paper 1 (Buzzard reference)         : b = {np.round(bias_paper_1, 4)}")

### The one-point functions

One panel per tomographic bin. The band is the member to member range of the flock, which is the only error bar
available here since DES Y3 is a single realization.

In [ ]:
fig, ax = plt.subplots(figsize=(15, 3.8), ncols=n_z, sharey=True)
for i in range(n_z):
    ax[i].plot(centres, target[i], color=colors["DES Y3"], lw=1.8, label="DES Y3", zorder=5)
    ax[i].plot(centres, cosmogrid["contam"]["stat"][i], color=colors["CosmoGrid"], lw=1.4, ls="--",
               label="CosmoGrid contaminated")
    for arm, label in (("raw", "Buzzard raw"), ("thinned", "Buzzard thinned"),
                       ("contaminated", "Buzzard contaminated")):
        stats = buzzard_arms[arm]["stat"][:, i]
        ax[i].fill_between(centres, stats.min(axis=0), stats.max(axis=0), color=colors[label], alpha=0.35, lw=0)
        ax[i].plot(centres, stats.mean(axis=0), color=colors[label], lw=1.2, label=label)
    ax[i].axvline(n_bar[i], color="C3", ls=":", lw=1)
    ax[i].set(xlabel="galaxies per pixel", xlim=(0, 180),
              title=f"{z_bins[i]}, CosmoGrid $b$ = {bias_fiducial['contam'][i]:.2f}")
    ax[i].grid(True, zorder=0)
ax[0].set(ylabel="pixel counts")
ax[0].legend(fontsize=7)
fig.suptitle("metacal source count one-point functions, footprint pixels")
fig.tight_layout()

### Widths and distance to the DES Y3 target

The Wasserstein distance is the loss `sc_bias_fit_count.ipynb` minimizes, so it is the honest one number summary of
how far a curve is from the target. Read it against the CosmoGrid rows, which are the floor: those curves are the
best fit of the statistic being tabulated, so whatever distance they still carry is what the model cannot represent
at any bias.

The widths say where the rest comes from. Clean Buzzard narrows with redshift while DES Y3 widens, because the
projected clustering weakens as the bins get radially broader and the imaging systematics do the opposite, their
standard deviation growing from 0.06 in `metacal1` to 0.22 in `metacal4`. Contaminating Buzzard is what makes
`metacal4` its widest bin rather than its narrowest, which is the ordering the data has.

In [ ]:
def row(label, mean, std, wd):
    return (f"  {label:26s}" + "".join(f"{m:7.1f} {s:6.1f} {w:7.2f}" for m, s, w in zip(mean, std, wd)))


print(" " * 28 + "".join(f"{z:^21s}" for z in z_bins))
print(" " * 28 + "".join(f"{'mean':>7s} {'std':>6s} {'W':>7s}" for _ in z_bins))
print(row("DES Y3 (the target)", n_bar, des_counts.std(axis=0), np.zeros(n_z)))
for group in cosmogrid:
    print(row(f"CosmoGrid {group}", cosmogrid[group]["mean"], cosmogrid[group]["std"], cosmogrid[group]["wd"]))
for arm in buzzard_arms:
    print(row(f"Buzzard {arm}", buzzard_arms[arm]["mean"].mean(axis=0),
              buzzard_arms[arm]["std"].mean(axis=0), buzzard_arms[arm]["wd"].mean(axis=0)))
print(row("Buzzard contam + Poisson", buzzard_pois["mean"].mean(axis=0),
          buzzard_pois["std"].mean(axis=0), buzzard_pois["wd"].mean(axis=0)))

# what the extra Poisson layer would add, which is why it is left out
predicted = np.sqrt(buzzard_arms["contaminated"]["std"].mean(axis=0) ** 2 + n_bar)
print("\nstd of the resampled variant, predicted as one extra Poisson layer:", np.round(predicted, 2))
print("                                                          measured:",
      np.round(buzzard_pois["std"].mean(axis=0), 2))

### What the difference costs in $b_{g,s}$

The widths only matter through the bias they imply, so the last step fits the fiducial cosmology against each
candidate reference with `scb.fit_bias`. Four arms, of which the first and the third are the two papers:

| arm | model | target |
|---|---|---|
| `buzzard_clean` | CosmoGrid as is | Buzzard thinned, clean (paper 1) |
| `buzzard_contam` | CosmoGrid $\times$ contamination | Buzzard thinned and contaminated |
| `desy3_contam` | CosmoGrid $\times$ contamination | DES Y3 (paper 2) |
| `desy3_clean` | CosmoGrid as is | DES Y3 |

The second arm is the control that separates the two effects. Contaminating *both* sides has to leave $b$ where the
clean pair put it, since the same map is applied to model and target; whatever is left between that and
`desy3_contam` is a genuine difference between the Buzzard source sample and DES Y3, not a systematics artifact.

`desy3_contam` has to reproduce the paper 2 table exactly, since that is the same fit. `buzzard_clean` is only
expected to come close to the paper 1 table: that one was fit with the permutation invariant loss of the reference
script, against a different realization of member 0, and without the thinning.

In [ ]:
raw_0 = buzzard.get_metacal_counts(flock_files[MEMBERS[0]])[mask]
thinned_0, _ = thin_to_des(raw_0, seed=MEMBERS[0])
buzzard_maps = {"clean": thinned_0, "contam": contaminate(thinned_0)}

# build_target wants a full sky map and asserts n_bar against the config, which the thinning is what makes pass
targets = {}
for name, counts in buzzard_maps.items():
    full = np.zeros((n_pix, n_z), dtype=np.float32)
    full[mask] = counts
    targets[f"buzzard_{name}"] = scb.build_target(conf, count_map=full)[:2]
targets["desy3_contam"] = (target, n_bar)
targets["desy3_clean"] = (target, n_bar)

fits = {}
for name, (tgt, nbar) in targets.items():
    contam = contamination if name.endswith("contam") else None
    fits[name] = scb.fit_bias(delta, tgt, nbar, mask=mask, kind="1pt", contamination=contam)
    print(f"{name:15s} b = {np.round(fits[name][0], 3)}   loss = {np.round(fits[name][1], 4)}")

print(f"\n{'paper 1 table':15s} b = {np.round(bias_paper_1, 3)}"
      f"   ratio to buzzard_clean {np.round(fits['buzzard_clean'][0] / bias_paper_1, 3)}")
print(f"{'paper 2 table':15s} b = {np.round(bias_fiducial['contam'], 3)}"
      f"   ratio to desy3_contam  {np.round(fits['desy3_contam'][0] / bias_fiducial['contam'], 3)}")
print(f"\nDES Y3 vs Buzzard as the reference, both contaminated: "
      f"{np.round(fits['desy3_contam'][0] / fits['buzzard_contam'][0], 3)}")
print(f"the contamination alone, Buzzard on both sides:        "
      f"{np.round(fits['buzzard_contam'][0] / fits['buzzard_clean'][0], 3)}")

The two ratios at the bottom are the result. The second one is the control and lands within a couple of percent of
1, confirming that a map applied to model and target alike cancels out of the fit, so the first ratio is not a
restatement of the contamination.

That first ratio is what changing the reference observation costs, and it is not a normalization. `metacal1` agrees,
and the other three bins grow steadily apart, so Buzzard and DES Y3 disagree about how much projected clustering the
source counts carry, increasingly with redshift and by much more than the imaging systematics account for. Which of
the two is right is not something this notebook can say. What it does say is that the paper 1 bias table cannot be
carried over, and that the fit had to be redone against the data rather than transplanted.